# Ground Condition Classifier — Prediction

Run this notebook to predict the ground condition for a **new raw CSV file**.

**Prerequisites:** Run `train_model.ipynb` first to generate:
- `kristian_model.pkl`
- `kristian_feature_cols.pkl`

**Steps in this notebook:**
1. Set `NEW_CSV_PATH` and optionally `TRUE_LABEL`
2. Run all cells — predictions and an output CSV are produced automatically

## 0. Configuration

Set the path to your CSV and (optionally) the true ground label.

In [2]:
# ── CONFIGURATION — edit these two lines ─────────────────────────
NEW_CSV_PATH = "Kristian_dry_0425.csv"   # path to the raw CSV to predict
TRUE_LABEL   = None                  # "Concrete" / "Brick" / "Grass"  or None if unknown
OUTPUT_CSV   = "predicted_results.csv"
# ─────────────────────────────────────────────────────────────────


## 1. Imports & Constants

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.signal import find_peaks, find_peaks as _find_peaks, savgol_filter
from scipy.stats import skew, kurtosis


In [4]:
# ── Sensor / signal constants (must match training) ───────────────
PRESSURE_COLS    = [f'pressure_{i:02d}' for i in range(1, 13)]
HEEL_SENSORS     = ['pressure_08', 'pressure_11']
FOREFOOT_SENSORS = ['pressure_02', 'pressure_04', 'pressure_05', 'pressure_06', 'pressure_09']
IMU_COLS         = ['accel_x', 'accel_y', 'accel_z',
                    'gyro_x',  'gyro_y',  'gyro_z',
                    'magn_x',  'magn_y',  'magn_z']

# Step-detection thresholds
SMOOTH_WIN    = 11
SMOOTH_POLY   = 2
BIG_PEAK_DIST = 50
BIG_PEAK_PROM = 200
BIG_PEAK_HT   = 500

# Loading-rate thresholds
HILL_WINDOW = 10
MIN_DP      = 5


## 2. Load Saved Model

In [5]:
# ── Load saved model and feature list ────────────────────────────
rf                 = joblib.load('kristian_model.pkl')
feature_cols_model = joblib.load('kristian_feature_cols.pkl')

print(f"Model loaded  : kristian_model.pkl  ({rf.n_estimators} trees)")
print(f"Feature count : {len(feature_cols_model)}")
print(f"Classes       : {list(rf.classes_)}")


Model loaded  : kristian_model.pkl  (200 trees)
Feature count : 115
Classes       : ['Brick', 'Concrete', 'Grass']


## 3. Load & Clean New CSV

In [6]:
# ── Load & clean the new CSV ──────────────────────────────────────
print(f"\n{'='*60}")
print(f"  Loading: {NEW_CSV_PATH}")

new_df = pd.read_csv(NEW_CSV_PATH)
new_df = new_df[new_df['corrupt'] == 0].copy()
new_df = new_df[new_df['sole_id'] == 1].reset_index(drop=True)

new_df['timestamp']      = new_df['timestamp'] - new_df['timestamp'].iloc[0]
new_df['time_sec']       = new_df['timestamp'] / 1000.0
new_df['total_pressure'] = new_df[PRESSURE_COLS].sum(axis=1)

print(f"  Rows after cleaning : {len(new_df):,}")
print(f"  Time span           : {new_df['time_sec'].iloc[-1]:.1f} s")



  Loading: Kristian_dry_0425.csv
  Rows after cleaning : 25,633
  Time span           : 410.1 s


## 4. Processing Functions

These are identical to the training notebook — do not modify them.

In [7]:
def find_step_windows(df):
    """Detect stance peaks from the smoothed combined heel signal.
    Returns array of peak indices — consecutive pairs define one step window."""
    y = df[HEEL_SENSORS].sum(axis=1).to_numpy(dtype=float)
    n  = len(y)
    sw = min(SMOOTH_WIN, n - (0 if n % 2 == 1 else 1))
    if sw % 2 == 0: sw -= 1
    if sw <= SMOOTH_POLY: sw = SMOOTH_POLY + 3 + (1 if (SMOOTH_POLY + 3) % 2 == 0 else 0)
    y_sm = savgol_filter(y, window_length=sw, polyorder=min(SMOOTH_POLY, sw - 1))
    peaks, _ = find_peaks(y_sm, distance=BIG_PEAK_DIST,
                           prominence=BIG_PEAK_PROM, height=BIG_PEAK_HT)
    return peaks


In [8]:
def compute_loading_rate(df, big_peaks):
    """
    For each step window:
      - contact (c)   = argmin of raw heel sensor signal in [p1, p2]
      - hill peak (h) = first local max within HILL_WINDOW samples of contact
      - loading_rate  = (pressure_h - pressure_c) / (time_h - time_c)  [ADC/s]
    Averaged across both heel sensors; step dropped if either sensor is invalid.
    """
    t       = df['time_sec'].values
    records = []

    for i in range(len(big_peaks) - 1):
        p1, p2    = big_peaks[i], big_peaks[i + 1]
        step_lrs  = []

        for sensor in HEEL_SENSORS:
            y = df[sensor].values.astype(float)

            c_local = int(np.argmin(y[p1:p2]))
            c_idx   = p1 + c_local

            end    = min(p2, c_idx + HILL_WINDOW)
            seg    = y[c_idx:end + 1]
            lp, _  = _find_peaks(seg, prominence=1)
            h_local = lp[0] if len(lp) else int(np.argmax(seg[1:])) + 1
            h_idx   = c_idx + h_local

            dt = t[h_idx] - t[c_idx]
            dp = y[h_idx] - y[c_idx]

            if dt > 0 and dp >= MIN_DP:
                step_lrs.append(dp / dt)

        if len(step_lrs) == len(HEEL_SENSORS):
            records.append({'step': i + 1, 'loading_rate': float(np.mean(step_lrs))})

    return pd.DataFrame(records)


In [9]:
def compute_midstance(df, big_peaks):
    """
    For each step window:
      - contact (c)         = argmin of combined heel signal in [p1, p2]
      - midstance           = first sample after c where forefoot sum > heel sum
      - fallback            = window midpoint (c + p2) // 2 if no crossover found
      - midstance_pressure  = sum of forefoot sensors at midstance index
      - time_to_midstance   = t[mid_idx] - t[c_idx]
    """
    heel_sig = df[HEEL_SENSORS].sum(axis=1).values.astype(float)
    fore_sig = df[FOREFOOT_SENSORS].sum(axis=1).values.astype(float)
    t        = df['time_sec'].values

    records = []
    for i in range(len(big_peaks) - 1):
        p1, p2  = big_peaks[i], big_peaks[i + 1]
        c_local = int(np.argmin(heel_sig[p1:p2]))
        c_idx   = p1 + c_local

        diff  = fore_sig[c_idx:p2] - heel_sig[c_idx:p2]
        cross = np.where(diff > 0)[0]

        if len(cross):
            mid_idx         = c_idx + int(cross[0])
            crossover_found = True
        else:
            mid_idx         = (c_idx + p2) // 2
            crossover_found = False

        records.append({
            'step':                i + 1,
            'midstance_pressure':  float(fore_sig[mid_idx]),
            'midstance_time_sec':  float(t[mid_idx]),
            'time_to_midstance':   float(t[mid_idx] - t[c_idx]),
            'crossover_found':     crossover_found,
        })

    return pd.DataFrame(records)


In [10]:
def compute_contact_times(df, big_peaks):
    """
    For each detected step window, finds heel-contact as the minimum of the
    combined heel signal and records the timestamp.
    """
    heel_sig = df[HEEL_SENSORS].sum(axis=1).values.astype(float)
    t = df['time_sec'].values

    rows = []
    for i in range(len(big_peaks) - 1):
        p1, p2 = big_peaks[i], big_peaks[i + 1]
        c_local = int(np.argmin(heel_sig[p1:p2]))
        c_idx = p1 + c_local
        rows.append({'step': i + 1, 'contact_time_sec': float(t[c_idx])})

    return pd.DataFrame(rows)


## 5. Step Segmentation

In [11]:
# ── Step segmentation ─────────────────────────────────────────────
big_peaks = find_step_windows(new_df)
n_steps   = max(0, len(big_peaks) - 1)
print(f"\n  Steps detected: {n_steps}")

if n_steps == 0:
    raise RuntimeError(
        "No steps detected — check that the CSV has valid heel pressure data."
    )



  Steps detected: 346


## 6. Feature Extraction

In [12]:
# ── Feature extraction (mirrors training exactly) ─────────────────
records = []
for k in range(n_steps):
    p1, p2 = big_peaks[k], big_peaks[k + 1]
    window = new_df.iloc[p1:p2]

    row = {
        'step':                k + 1,
        'step_duration':       len(window),
        'total_pressure_mean': window['total_pressure'].mean(),
        'stance_auc':          np.trapezoid(window['total_pressure'].values,
                                            window['time_sec'].values),
    }
    for col in PRESSURE_COLS + IMU_COLS:
        x = window[col].values.astype(float)
        row[f'{col}_mean']     = x.mean()
        row[f'{col}_std']      = x.std()
        row[f'{col}_max']      = x.max()
        row[f'{col}_min']      = x.min()
        row[f'{col}_skew']     = skew(x)
        row[f'{col}_kurtosis'] = kurtosis(x)
    records.append(row)

new_steps = pd.DataFrame(records)

# Loading rate
lr_df     = compute_loading_rate(new_df, big_peaks)
new_steps = new_steps.merge(lr_df[['step', 'loading_rate']], on='step', how='left')

# Midstance
mid_df    = compute_midstance(new_df, big_peaks)
new_steps = new_steps.merge(
    mid_df[['step', 'midstance_pressure', 'midstance_time_sec',
            'time_to_midstance', 'crossover_found']],
    on='step', how='left')

# Contact times
contact_df = compute_contact_times(new_df, big_peaks)
new_steps  = new_steps.merge(contact_df, on='step', how='left')

print(f"  Feature matrix: {new_steps.shape}")


  Feature matrix: (346, 136)


## 7. Predict

In [13]:
# ── Align to training feature columns & predict ───────────────────
for col in feature_cols_model:
    if col not in new_steps.columns:
        new_steps[col] = 0.0
        print(f"  [WARN] Missing feature '{col}' — filled with 0")

X_new      = new_steps[feature_cols_model].apply(pd.to_numeric, errors='coerce').fillna(0)
y_pred_new = rf.predict(X_new)
new_steps['predicted_ground_type'] = y_pred_new

prob_df = pd.DataFrame(rf.predict_proba(X_new),
                       columns=[f'prob_{c}' for c in rf.classes_])
new_steps = pd.concat([new_steps.reset_index(drop=True), prob_df], axis=1)

print(f"\n  Prediction summary:")
for label, cnt in pd.Series(y_pred_new).value_counts().items():
    print(f"    {label:<12}: {cnt:3d} steps  ({cnt/len(y_pred_new)*100:.1f}%)")



  Prediction summary:
    Concrete    : 211 steps  (61.0%)
    Brick       : 133 steps  (38.4%)
    Grass       :   2 steps  (0.6%)


## 8. Accuracy & Confusion Matrix

Shown only when `TRUE_LABEL` is set. If `TRUE_LABEL = None`, per-step class probabilities are printed instead.

In [14]:
# ── Accuracy & confusion matrix (only when TRUE_LABEL is known) ───
if TRUE_LABEL is not None:
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

    y_true_new = [TRUE_LABEL] * len(y_pred_new)
    acc = accuracy_score(y_true_new, y_pred_new)

    print(f"\nAccuracy: {acc:.3f}  ({acc*100:.1f}%)")
    print("\nClassification Report:")
    print(classification_report(y_true_new, y_pred_new))

    all_labels = sorted(rf.classes_)
    cm_new = confusion_matrix(y_true_new, y_pred_new, labels=all_labels)

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_new, annot=True, fmt='d', cmap='Blues',
                xticklabels=all_labels, yticklabels=all_labels, ax=ax)
    ax.set_title(f'Confusion Matrix\n(True label: {TRUE_LABEL})')
    ax.set_ylabel('True')
    ax.set_xlabel('Predicted')
    plt.tight_layout()
    plt.show()

else:
    print("\n  (TRUE_LABEL=None — skipping accuracy metrics)")
    print("\n  Per-step probabilities:")
    prob_cols = [f'prob_{c}' for c in rf.classes_]
    header = f"  {'Step':>4}  {'Predicted':<12}  " + "  ".join(f"{c}" for c in rf.classes_)
    print(header)
    for _, row in new_steps.iterrows():
        probs = "  ".join(f"{row[p]*100:5.1f}%" for p in prob_cols)
        print(f"  {int(row['step']):>4}  {row['predicted_ground_type']:<12}  {probs}")



  (TRUE_LABEL=None — skipping accuracy metrics)

  Per-step probabilities:
  Step  Predicted     Brick  Concrete  Grass
     1  Grass          28.5%   33.0%   38.5%
     2  Concrete       32.0%   36.5%   31.5%
     3  Brick          42.0%   28.0%   30.0%
     4  Brick          34.5%   33.0%   32.5%
     5  Concrete       33.0%   35.5%   31.5%
     6  Concrete       32.5%   35.0%   32.5%
     7  Grass          33.0%   32.5%   34.5%
     8  Brick          35.0%   32.5%   32.5%
     9  Brick          37.0%   32.5%   30.5%
    10  Brick          48.0%   38.5%   13.5%
    11  Brick          49.5%   38.0%   12.5%
    12  Brick          50.0%   36.5%   13.5%
    13  Brick          41.0%   36.0%   23.0%
    14  Brick          45.5%   40.0%   14.5%
    15  Brick          43.5%   39.0%   17.5%
    16  Brick          42.5%   36.5%   21.0%
    17  Brick          42.5%   38.0%   19.5%
    18  Brick          46.0%   37.5%   16.5%
    19  Brick          42.0%   32.5%   25.5%
    20  Brick          4

## 9. Save Results CSV

In [15]:
# ── Save results CSV ──────────────────────────────────────────────
out_cols = (
    ['step', 'predicted_ground_type']
    + [f'prob_{c}' for c in rf.classes_]
    + ['contact_time_sec', 'loading_rate', 'midstance_pressure',
       'time_to_midstance', 'step_duration', 'stance_auc']
)
out_cols  = [c for c in out_cols if c in new_steps.columns]
output_df = new_steps[out_cols].copy()

if TRUE_LABEL:
    output_df.insert(2, 'true_ground_type', TRUE_LABEL)

output_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved → {OUTPUT_CSV}")
print(output_df.head())



Saved → predicted_results.csv
   step predicted_ground_type  prob_Brick  prob_Concrete  prob_Grass  \
0     1                 Grass       0.285          0.330       0.385   
1     2              Concrete       0.320          0.365       0.315   
2     3                 Brick       0.420          0.280       0.300   
3     4                 Brick       0.345          0.330       0.325   
4     5              Concrete       0.330          0.355       0.315   

   contact_time_sec  loading_rate  midstance_pressure  time_to_midstance  \
0             3.568   1602.083333              1469.0                0.0   
1             4.720   1078.125000              1856.0                0.0   
2             5.856    914.062500              1975.0                0.0   
3             7.056    953.571429              1649.0                0.0   
4             8.256   1906.250000              1688.0                0.0   

   step_duration  stance_auc  
0             76    4118.248  
1             77 

## 10. Step Timeline

Chronological view of every predicted step.

In [16]:
# ── Timeline print ────────────────────────────────────────────────
timeline_df = new_steps[[
    'step', 'predicted_ground_type',
    'contact_time_sec', 'midstance_time_sec',
    'loading_rate', 'midstance_pressure'
]].copy()

if TRUE_LABEL:
    timeline_df.insert(2, 'ground_type', TRUE_LABEL)

timeline_df = timeline_df.sort_values('contact_time_sec').reset_index(drop=True)
timeline_df['walk_progress_pct'] = (timeline_df.index + 1) / len(timeline_df) * 100

with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                       'display.width', None):
    print(timeline_df.to_string(index=False))


 step predicted_ground_type  contact_time_sec  midstance_time_sec  loading_rate  midstance_pressure  walk_progress_pct
    1                 Grass             3.568               3.568   1602.083333              1469.0           0.289017
    2              Concrete             4.720               4.720   1078.125000              1856.0           0.578035
    3                 Brick             5.856               5.856    914.062500              1975.0           0.867052
    4                 Brick             7.056               7.056    953.571429              1649.0           1.156069
    5              Concrete             8.256               8.256   1906.250000              1688.0           1.445087
    6              Concrete             9.552               9.552   1218.750000              1715.0           1.734104
    7                 Grass            10.816              10.816   1312.500000              1854.0           2.023121
    8                 Brick            12.048   

## 11. Animated Walk Progress (optional)

Interactive Plotly animation — shows predicted ground condition step by step. Requires `TRUE_LABEL` to be set for the true-ground annotation row.

In [17]:
import plotly.graph_objects as go

n_steps = len(timeline_df)

frames = []
for i in range(n_steps):
    current = timeline_df.iloc[i]
    progress = current['walk_progress_pct']

    # show last few steps as history
    history_start = max(0, i - 4)
    recent = timeline_df.iloc[history_start:i+1].copy()

    recent_text = "<br>".join(
        [
            f"Step {idx+1}: true={row['ground_type']} | pred={row['predicted_ground_type']}"
            for idx, row in recent.iterrows()
        ]
    )

    frame = go.Frame(
        name=str(i),
        data=[
            # progress bar
            go.Bar(
                x=[progress],
                y=["Walk Progress"],
                orientation="h",
                width=[0.45],
                text=[f"{progress:.1f}%"],
                textposition="inside"
            )
        ],
        layout=go.Layout(
            annotations=[
                dict(
                    x=0.5, y=0.95, xref="paper", yref="paper",
                    text=f"<b>Step {i+1} of {n_steps}</b>",
                    showarrow=False,
                    font=dict(size=22)
                ),
                dict(
                    x=0.5, y=0.78, xref="paper", yref="paper",
                    text=f"Predicted ground: <b>{current['predicted_ground_type']}</b>",
                    showarrow=False,
                    font=dict(size=20)
                ),
                dict(
                    x=0.5, y=0.68, xref="paper", yref="paper",
                    text=f"True ground: <b>{current['ground_type']}</b>",
                    showarrow=False,
                    font=dict(size=18)
                ),
                dict(
                    x=0.5, y=0.58, xref="paper", yref="paper",
                    text=f"Contact time: {current['contact_time_sec']:.2f} s",
                    showarrow=False,
                    font=dict(size=16)
                ),
                dict(
                    x=0.5, y=0.48, xref="paper", yref="paper",
                    text=f"Loading rate: {current['loading_rate']:.1f}" if pd.notna(current['loading_rate']) else "Loading rate: NA",
                    showarrow=False,
                    font=dict(size=16)
                ),
                dict(
                    x=0.5, y=0.18, xref="paper", yref="paper",
                    text=f"Recent steps:<br>{recent_text}",
                    showarrow=False,
                    align="left",
                    font=dict(size=14)
                )
            ]
        )
    )
    frames.append(frame)

# Initial frame
first = timeline_df.iloc[0]

fig = go.Figure(
    data=[
        go.Bar(
            x=[first['walk_progress_pct']],
            y=["Walk Progress"],
            orientation="h",
            width=[0.45],
            text=[f"{first['walk_progress_pct']:.1f}%"],
            textposition="inside"
        )
    ],
    layout=go.Layout(
        title="Walk Progress and Ground Condition by Step",
        xaxis=dict(range=[0, 100], title="Percent of walk completed"),
        yaxis=dict(showticklabels=True),
        template="plotly_white",
        height=700,
        updatemenus=[
            {
                "type": "buttons",
                "showactive": False,
                "buttons": [
                    {
                        "label": "Play",
                        "method": "animate",
                        "args": [None, {
                            "frame": {"duration": 500, "redraw": True},
                            "transition": {"duration": 200},
                            "fromcurrent": True
                        }]
                    },
                    {
                        "label": "Pause",
                        "method": "animate",
                        "args": [[None], {
                            "frame": {"duration": 0, "redraw": False},
                            "mode": "immediate",
                            "transition": {"duration": 0}
                        }]
                    }
                ]
            }
        ],
        annotations=[
            dict(
                x=0.5, y=0.95, xref="paper", yref="paper",
                text=f"<b>Step 1 of {n_steps}</b>",
                showarrow=False,
                font=dict(size=22)
            ),
            dict(
                x=0.5, y=0.78, xref="paper", yref="paper",
                text=f"Predicted ground: <b>{first['predicted_ground_type']}</b>",
                showarrow=False,
                font=dict(size=20)
            ),
            dict(
                x=0.5, y=0.68, xref="paper", yref="paper",
                text=f"True ground: <b>{first['ground_type']}</b>",
                showarrow=False,
                font=dict(size=18)
            ),
            dict(
                x=0.5, y=0.58, xref="paper", yref="paper",
                text=f"Contact time: {first['contact_time_sec']:.2f} s",
                showarrow=False,
                font=dict(size=16)
            ),
            dict(
                x=0.5, y=0.48, xref="paper", yref="paper",
                text=f"Loading rate: {first['loading_rate']:.1f}" if pd.notna(first['loading_rate']) else "Loading rate: NA",
                showarrow=False,
                font=dict(size=16)
            )
        ]
    ),
    frames=frames
)

fig.show()

KeyError: 'ground_type'